# Poke Agent Unified Run

Run this notebook end-to-end.

- On Kaggle/Linux with CABT `cg-lib` available, it can generate rollout data.
- On this Mac, it uses existing rollout JSONL data and trains with Torch on Apple Silicon MPS.
- It does not submit to the competition leaderboard.


In [1]:
from __future__ import annotations

import glob
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

print("repo", ROOT)
print("python", sys.version.split()[0])
print("torch", torch.__version__)


repo /Users/tsinzitari/Documents/poke-agent
python 3.11.15
torch 2.12.1


In [2]:
def torch_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = torch_device()
print("device", DEVICE)


device mps


In [3]:
def find_cg_lib() -> str | None:
    candidates: list[str] = []
    if os.environ.get("CG_LIB_PATH"):
        candidates.append(os.environ["CG_LIB_PATH"])
    candidates.extend(glob.glob("/kaggle/input/**/cg-lib", recursive=True))
    candidates.extend(glob.glob(str(ROOT / "kaggle/input/**/cg-lib"), recursive=True))
    return candidates[0] if candidates else None

CG_LIB_PATH = find_cg_lib()
CG_AVAILABLE = False
CG_ERROR = None
if CG_LIB_PATH:
    sys.path.append(CG_LIB_PATH)
    try:
        from cg.game import battle_finish, battle_select, battle_start
        from cg.api import to_observation_class
        CG_AVAILABLE = True
    except Exception as exc:
        CG_ERROR = repr(exc)

print("cg_lib_path", CG_LIB_PATH)
print("cg_available", CG_AVAILABLE)
if CG_ERROR:
    print("cg_error", CG_ERROR)


cg_lib_path /Users/tsinzitari/Documents/poke-agent/kaggle/input/cg-lib
cg_available False
cg_error OSError("dlopen(/Users/tsinzitari/Documents/poke-agent/kaggle/input/cg-lib/cg/libcg.so, 0x0006): tried: '/Users/tsinzitari/Documents/poke-agent/kaggle/input/cg-lib/cg/libcg.so' (slice is not valid mach-o file), '/System/Volumes/Preboot/Cryptexes/OS/Users/tsinzitari/Documents/poke-agent/kaggle/input/cg-lib/cg/libcg.so' (no such file), '/Users/tsinzitari/Documents/poke-agent/kaggle/input/cg-lib/cg/libcg.so' (slice is not valid mach-o file)")


In [4]:
SAMPLE_DECK = [
    721, 721, 722, 722, 722, 722, 723, 723, 723, 723,
    1092, 1121, 1121, 1145, 1145, 1163, 1163,
    1219, 1219, 1219, 1219, 1227, 1227, 1227, 1227,
    1262, 1262,
    3, 3, 3, 3, 3, 3, 3, 3, 3,
    3, 3, 3, 3, 3, 3, 3, 3, 3,
    3, 3, 3, 3, 3, 3, 3, 3, 3,
    3, 3, 3, 3, 3, 3,
]

def read_deck() -> list[int]:
    for path in [ROOT / "submission/deck.csv", ROOT / "deck.csv", Path("/kaggle_simulations/agent/deck.csv")]:
        if path.exists():
            deck = [int(line.strip()) for line in path.read_text().splitlines() if line.strip()]
            if len(deck) != 60:
                raise ValueError(f"{path} must contain 60 card IDs")
            return deck
    return SAMPLE_DECK

DECK = read_deck()
print("deck cards", len(DECK))


deck cards 60


In [5]:
def random_agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    options = list(range(len(obs.select.option)))
    return random.sample(options, min(obs.select.maxCount, len(options)))


def features_from_observation(obs: dict) -> list[float]:
    current = obs.get("current") or {}
    players = current.get("players") or [{}, {}]
    p0 = players[0] if len(players) > 0 else {}
    p1 = players[1] if len(players) > 1 else {}
    select = obs.get("select") or {}
    return [
        float(current.get("turn", 0)),
        float(current.get("yourIndex", 0)),
        float(p0.get("deckCount", 0)),
        float(p0.get("handCount", 0)),
        float(len(p0.get("bench", []))),
        float(p1.get("deckCount", 0)),
        float(p1.get("handCount", 0)),
        float(len(p1.get("bench", []))),
        float(len(select.get("option", []))),
        float(select.get("maxCount", 0)),
    ]


def play_episode(episode: int, max_steps: int = 300) -> list[dict]:
    rows = []
    obs, start_data = battle_start(DECK, DECK)
    if start_data.errorPlayer >= 0:
        raise ValueError(f"deck error type={start_data.errorType} player={start_data.errorPlayer}")
    try:
        step = 0
        while obs["current"]["result"] < 0 and step < max_steps:
            rows.append({
                "episode": episode,
                "step": step,
                "features": features_from_observation(obs),
                "player": int(obs["current"]["yourIndex"]),
            })
            obs = battle_select(random_agent(obs))
            step += 1
        result = int(obs["current"]["result"])
        for row in rows:
            row["value"] = 0.0 if result == 2 else (1.0 if row["player"] == result else -1.0)
        return rows
    finally:
        battle_finish()


In [6]:
GENERATE_EPISODES = int(os.environ.get("CABT_EPISODES", "3" if CG_AVAILABLE else "0"))
GENERATED_PATH = ROOT / "data/notebook_rollouts.jsonl"

if CG_AVAILABLE and GENERATE_EPISODES > 0:
    GENERATED_PATH.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for episode in range(GENERATE_EPISODES):
        rows.extend(play_episode(episode))
    with GENERATED_PATH.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, separators=(",", ":")) + "\n")
    print(f"generated {len(rows)} rows -> {GENERATED_PATH}")
else:
    print("skipping CABT generation in this runtime")


skipping CABT generation in this runtime


In [7]:
DATA_CANDIDATES = [
    ROOT / "data/notebook_rollouts.jsonl",
    ROOT / "data/kaggle-output/data/cabt_rollouts.jsonl",
    ROOT / "data/container-smoke.jsonl",
]

def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    print("No rollout data found. Using synthetic smoke data so Run All still completes.")
    rng = np.random.default_rng(7)
    x_np = rng.normal(size=(128, 10)).astype(np.float32)
    y_np = np.tanh(x_np[:, 0] * 0.1 + x_np[:, 2] * 0.03 - x_np[:, 5] * 0.03).astype(np.float32)
else:
    rows = load_jsonl(DATA_PATH)
    x_np = np.array([row["features"] for row in rows], dtype=np.float32)
    y_np = np.array([row["value"] for row in rows], dtype=np.float32)
    print(f"loaded {len(rows)} rows from {DATA_PATH}")

x = torch.tensor(x_np, device=DEVICE)
y = torch.tensor(y_np, device=DEVICE)
print("x", tuple(x.shape), "y", tuple(y.shape))


loaded 715 rows from /Users/tsinzitari/Documents/poke-agent/data/kaggle-output/data/cabt_rollouts.jsonl
x (715, 10) y (715,)


In [8]:
class ValueModel(torch.nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

model = ValueModel(x.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

EPOCHS = int(os.environ.get("TRAIN_EPOCHS", "50"))
for epoch in range(EPOCHS):
    optimizer.zero_grad(set_to_none=True)
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    if epoch in {0, EPOCHS - 1}:
        print(f"epoch={epoch + 1} loss={loss.item():.5f}")


epoch=1 loss=18.22668


epoch=500 loss=0.62261


In [9]:
OUT = ROOT / "out/value_model.pt"
OUT.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": x.shape[1],
    "device_used": str(DEVICE),
    "data_path": str(DATA_PATH) if DATA_PATH else None,
}, OUT)
print("saved", OUT)


saved /Users/tsinzitari/Documents/poke-agent/out/value_model.pt
